# State-Filtered Deliberation (SFD) Developer Agent
### Official Submission for The Gemma 4 Developer Agent Competition
**Author:** Karandeep Singh Sodhi (karandeep.sodhi2504a@gmail.com)
**Paper Writeup:** *State-Filtered Deliberation: Mitigating Context Bloat for Small Open-Weight Developer Agents*

This notebook packages the complete, zero-dependency declarative `submission.zip` archive implementing **State-Filtered Deliberation (SFD)**.


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
AGENT_DIR = WORKING_DIR / "agent_sfd"
AGENT_DIR.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / "configs").mkdir(exist_ok=True)
(AGENT_DIR / "prompts").mkdir(exist_ok=True)
(AGENT_DIR / "sub_agents").mkdir(exist_ok=True)

print("Building SFD agent structure...")


In [ ]:
agent_yaml_content = 'name: sfd_sequential_agent\nagent_class: SequentialAgent\ndescription: State-Filtered Deliberation (SFD) hierarchical agent executing triage, surgical patching, and verification without context bloat.\nsub_agents:\n  - config_path: sub_agents/triage_agent.yaml\n  - config_path: sub_agents/patch_agent.yaml\n  - config_path: sub_agents/verifier_agent.yaml\n'
eval_cfg_content = 'evaluation:\n  timeout_seconds: 120\n  max_tool_calls: 25\n  max_time_minutes: 10\n  max_turns: 40\n'
sampling_yaml_content = 'temperature: 0.1\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 2048\n  include_thoughts: true\n'
triage_agent_content = 'name: triage_agent\ndescription: Pinpoints the exact root-cause file and symbol without reading entire files.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/triage.md\ntools:\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\noutput_key: triage_packet\ngenerate_content_config: !include ../configs/sampling.yaml\n'
patch_agent_content = 'name: patch_agent\ndescription: Applies surgical code modifications to resolve the localized defect.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/patcher.md\ninclude_contents: "none"\ntools:\n  - read_file\n  - edit_file\n  - write_file\noutput_key: patch_summary\ngenerate_content_config: !include ../configs/sampling.yaml\n'
verifier_agent_content = 'name: verifier_agent\ndescription: Executes targeted verification and calls submit_patch.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/verifier.md\ninclude_contents: "none"\ntools:\n  - run_command\n  - submit_patch\n  - get_status\ngenerate_content_config: !include ../configs/sampling.yaml\n'
triage_prompt_content = 'You are a specialized Software Architecture Localization Agent.\nYour sole mission is to identify the precise file and function responsible for the reported issue in the minimum number of steps.\n\n## Problem Statement:\n{problem_description}\n\n## Protocol:\n1. Extract key symbols, class names, function names, or error messages from the problem description.\n2. If explicit filenames are provided, verify them using `search_similar_code` or `get_code_neighbors`.\n3. If no filenames are given, query `search_similar_code` with the exact function/method name or error keywords.\n4. Read only the specific target slice using `read_file` with explicit `start_line` and `end_line` bounds. Do not read entire files.\n5. Identify the exact file path, the function/class name, the line range, and the relevant targeted test path if mentioned.\n\n## Final Output Format:\nEmit a clean, compact YAML triage packet inside a `<triage_packet>` tag:\n\n<triage_packet>\ntarget_file: "path/to/file.py"\ntarget_symbol: "function_or_class_name"\nline_range: [start_line, end_line]\nsummary: "1-2 sentence description of the bug and the required fix"\ntest_target: "path/to/test_file.py -k test_name" # if identifiable\n</triage_packet>\n\nConclude immediately after emitting the triage packet.\n'
patcher_prompt_content = 'You are a Precision Software Engineering Patching Agent.\nYour sole responsibility is to apply the minimal, surgical code modification required to resolve the issue.\n\n## Problem Statement:\n{problem_description}\n\n## Localization Triage Information:\n{triage_packet}\n\n## Operating Rules:\n1. Examine the `target_file` identified in the triage information.\n2. Call `read_file` on `target_file` restricting strictly to the relevant `line_range`.\n3. Plan the minimal, necessary code change.\n4. Call `edit_file` with the exact `old_string` and `new_string` to apply the fix. Ensure indentation matches the existing codebase perfectly.\n5. Do NOT touch, modify, or create test files. All modifications must be strictly in source code files.\n6. Do NOT refactor or reformat unrelated code.\n\n## Final Output Format:\nEmit a brief summary of the applied modification inside `<patch_summary>`:\n\n<patch_summary>\nfile_modified: "path/to/file.py"\nstatus: "APPLIED"\ndescription: "Brief explanation of the logic change"\n</patch_summary>\n'
verifier_prompt_content = 'You are a Software Quality & Verification Agent.\nYour mission is to verify that the applied code patch solves the issue without regressions, and then submit the final patch.\n\n## Context:\nTriage:\n{triage_packet}\n\nPatch Status:\n{patch_summary}\n\n## Instructions:\n1. Call `run_command` with `git status` to ensure only the intended source files are modified.\n2. If any scratch or reproduction files were created in `/workspace/`, remove them immediately (temporary scripts must be in `/tmp/` to avoid polluting the patch).\n3. If a targeted test was identified, execute it using `run_command` (e.g. `pytest tests/path_to_test.py -k test_target`).\n   - If tests fail, diagnose whether it is an issue with your patch or an unrelated pre-existing failure.\n   - Do NOT run bare `pytest` across the entire repository.\n4. Once verified, call `submit_patch` to stage and record the unified git diff.\n5. Inspect the response of `submit_patch` to ensure `patch_size > 0` and `files_changed > 0`.\n6. Output a final sentence confirming patch submission to finish the session.\n'

# Write files to AGENT_DIR
(AGENT_DIR / "agent.yaml").write_text(agent_yaml_content, encoding="utf-8")
(AGENT_DIR / "eval_config.yaml").write_text(eval_cfg_content, encoding="utf-8")
(AGENT_DIR / "configs" / "sampling.yaml").write_text(sampling_yaml_content, encoding="utf-8")
(AGENT_DIR / "sub_agents" / "triage_agent.yaml").write_text(triage_agent_content, encoding="utf-8")
(AGENT_DIR / "sub_agents" / "patch_agent.yaml").write_text(patch_agent_content, encoding="utf-8")
(AGENT_DIR / "sub_agents" / "verifier_agent.yaml").write_text(verifier_agent_content, encoding="utf-8")
(AGENT_DIR / "prompts" / "triage.md").write_text(triage_prompt_content, encoding="utf-8")
(AGENT_DIR / "prompts" / "patcher.md").write_text(patcher_prompt_content, encoding="utf-8")
(AGENT_DIR / "prompts" / "verifier.md").write_text(verifier_prompt_content, encoding="utf-8")

print("Declarative agent tree written successfully.")


In [ ]:
# Package into /kaggle/working/submission.zip with agent.yaml at root
zip_base = WORKING_DIR / "submission"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=AGENT_DIR))

# Validate archive contents
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    assert "agent.yaml" in names, "Missing agent.yaml at root of submission.zip!"
    print(f"Created {zip_path.name} ({zip_path.stat().st_size} bytes)")
    print("Archive entries:")
    for n in sorted(names):
        print(f"  - {n}")
